# Тренажёр для live-кодинга на Python (Data Science / ML)

Ноутбук для отработки навыков перед техническим интервью / live-coding сессией.

**Темы:**
1. Препроцессинг данных
2. Feature Engineering
3. Регрессия
4. Бинарная классификация
5. Мультиклассовая классификация
6. Кластеризация

**Как пользоваться:**
- Каждая задача — отдельная markdown-ячейка с условием + кодовая ячейка с заготовкой (`# TODO`).
- Данные генерируются синтетически внутри ноутбука — интернет не нужен.
- В конце каждого раздела — ячейка `# РЕШЕНИЕ (скрыто)` с одним из вариантов решения. Старайтесь сначала решить сами, потом сверяйтесь.
- Рекомендуемый тайминг на live-coding интервью: 10-20 минут на задачу.


In [ ]:
# Общие импорты — выполните эту ячейку первой
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, KFold
from sklearn.preprocessing import (StandardScaler, MinMaxScaler, OneHotEncoder,
                                    OrdinalEncoder, PolynomialFeatures, LabelEncoder)
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingClassifier
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import (mean_squared_error, mean_absolute_error, r2_score,
                              accuracy_score, precision_score, recall_score, f1_score,
                              roc_auc_score, confusion_matrix, classification_report,
                              silhouette_score, davies_bouldin_score)
from sklearn.datasets import make_regression, make_classification, make_blobs

import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Готово. Все библиотеки импортированы.")


---
# Раздел 1. Препроцессинг данных


## Задача 1.1 — Пропуски и типы данных

Дан датасет о клиентах интернет-магазина с пропусками, "грязными" типами и выбросами.

**Требуется:**
1. Найти и вывести количество/долю пропусков по каждому столбцу.
2. Обработать пропуски: `age` — медианой, `income` — KNNImputer или группой по `city`, `category` — модой или отдельной категорией "unknown".
3. Привести `signup_date` к типу datetime.
4. Найти и обработать выбросы в `income` (например, методом IQR).


In [ ]:
# Генерация "грязных" данных
n = 500
df = pd.DataFrame({
    "customer_id": range(1, n + 1),
    "age": np.random.normal(35, 12, n).round(),
    "income": np.random.lognormal(mean=10.5, sigma=0.6, size=n),
    "city": np.random.choice(["Moscow", "SPb", "Kazan", "Novosibirsk", None], n, p=[0.4, 0.25, 0.15, 0.15, 0.05]),
    "category": np.random.choice(["electronics", "clothes", "food", "books"], n),
    "signup_date": pd.date_range("2021-01-01", periods=n, freq="17h").astype(str),
})

# Внесём пропуски и выбросы искусственно
df.loc[np.random.choice(n, 40, replace=False), "age"] = np.nan
df.loc[np.random.choice(n, 60, replace=False), "income"] = np.nan
df.loc[np.random.choice(n, 25, replace=False), "category"] = np.nan
df.loc[np.random.choice(n, 5, replace=False), "income"] *= 15  # выбросы
df.loc[np.random.choice(n, 5, replace=False), "age"] = -5      # невалидные значения

df.head()


In [ ]:
# TODO 1.1.1: посчитайте количество и долю пропусков по каждому столбцу


# TODO 1.1.2: обработайте пропуски в age (медиана), category (мода/'unknown'),
#             income (KNNImputer или группировка по city)


# TODO 1.1.3: приведите signup_date к datetime


# TODO 1.1.4: найдите выбросы в income методом IQR и решите, что с ними делать
#             (обрезать/заменить на границы/удалить строки)


## Задача 1.2 — Кодирование категориальных признаков

**Требуется:**
1. Закодировать `city` через One-Hot Encoding.
2. Закодировать `category` через Ordinal Encoding (придумайте логичный порядок) — или Target Encoding по `income` (среднее income по категории), избегая утечки данных (посчитать target encoding только на train).
3. Сравнить размерность датафрейма до и после кодирования.


In [ ]:
# TODO 1.2.1: One-Hot Encoding для city (используйте pd.get_dummies или OneHotEncoder)


# TODO 1.2.2: Ordinal или Target Encoding для category.
#             Если Target Encoding — сначала train_test_split, считать среднее ТОЛЬКО на train!


# TODO 1.2.3: сравните df.shape до/после


## Задача 1.3 — Масштабирование признаков

**Требуется:** обучить `StandardScaler` на train-выборке и применить (не переобучая!) к train и test.
Объяснить устно/в комментарии, почему нельзя фитить scaler на всём датасете сразу.


In [ ]:
# TODO 1.3: разбейте данные на train/test, зафитите StandardScaler на train,
#            примените transform к train и test


In [ ]:
# ===================== РЕШЕНИЕ раздела 1 (скрыто) =====================
'''
# 1.1
missing = df.isna().sum()
missing_pct = (df.isna().mean() * 100).round(2)
print(pd.DataFrame({"count": missing, "pct": missing_pct}))

df["age"] = df["age"].mask(df["age"] < 0)  # невалидные -> NaN
df["age"] = df["age"].fillna(df["age"].median())

df["category"] = df["category"].fillna("unknown")

imputer = KNNImputer(n_neighbors=5)
df[["income"]] = imputer.fit_transform(df[["income"]])

df["signup_date"] = pd.to_datetime(df["signup_date"])

q1, q3 = df["income"].quantile([0.25, 0.75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
df["income"] = df["income"].clip(lower, upper)

# 1.2
df_ohe = pd.get_dummies(df, columns=["city"], prefix="city")

order = [["unknown", "food", "books", "clothes", "electronics"]]
oe = OrdinalEncoder(categories=order)
df["category_ord"] = oe.fit_transform(df[["category"]])

# 1.3
train_df, test_df = train_test_split(df, test_size=0.2, random_state=RANDOM_STATE)
scaler = StandardScaler()
train_scaled = scaler.fit_transform(train_df[["age", "income"]])
test_scaled = scaler.transform(test_df[["age", "income"]])
'''


---
# Раздел 2. Feature Engineering


## Задача 2.1 — Признаки из даты

Используя `signup_date` из раздела 1, создайте признаки: `signup_year`, `signup_month`, `signup_dayofweek`,
`is_weekend`, `days_since_signup` (относительно "сегодня" = 2024-01-01).


In [ ]:
# TODO 2.1: создайте признаки из даты signup_date


## Задача 2.2 — Взаимодействия и полиномиальные признаки

Дан набор числовых признаков `x1, x2, x3`. Постройте:
1. Попарные произведения признаков (`x1*x2`, `x1*x3`, `x2*x3`).
2. Полиномиальные признаки 2-й степени через `PolynomialFeatures` (без утечки — fit только на train).
3. Биннинг (дискретизацию) признака `x1` на 4 квантильных бина через `pd.qcut`.


In [ ]:
num_df = pd.DataFrame({
    "x1": np.random.normal(0, 1, 300),
    "x2": np.random.normal(5, 2, 300),
    "x3": np.random.exponential(2, 300),
})
num_df.head()


In [ ]:
# TODO 2.2.1: попарные произведения признаков


# TODO 2.2.2: PolynomialFeatures degree=2 (fit на train, transform на train/test)


# TODO 2.2.3: биннинг x1 на 4 квантильных бина (pd.qcut), закодировать бины числом/OHE


## Задача 2.3 — Агрегирующие признаки (groupby)

Дан датасет транзакций `customer_id, amount, category`. Постройте для каждого `customer_id` признаки:
`total_spent`, `avg_check`, `n_transactions`, `n_unique_categories`, `max_amount`.
Присоедините полученную таблицу обратно к исходному датафрейму клиентов по `customer_id`.


In [ ]:
tx = pd.DataFrame({
    "customer_id": np.random.choice(range(1, 101), 2000),
    "amount": np.random.gamma(2, 50, 2000).round(2),
    "category": np.random.choice(["electronics", "clothes", "food", "books"], 2000),
})
tx.head()


In [ ]:
# TODO 2.3: постройте агрегаты по customer_id через groupby/agg
#            и объедините с таблицей клиентов (merge по customer_id)


In [ ]:
# ===================== РЕШЕНИЕ раздела 2 (скрыто) =====================
'''
# 2.1
today = pd.Timestamp("2024-01-01")
df["signup_year"] = df["signup_date"].dt.year
df["signup_month"] = df["signup_date"].dt.month
df["signup_dayofweek"] = df["signup_date"].dt.dayofweek
df["is_weekend"] = df["signup_dayofweek"].isin([5, 6]).astype(int)
df["days_since_signup"] = (today - df["signup_date"]).dt.days

# 2.2
num_df["x1_x2"] = num_df["x1"] * num_df["x2"]
num_df["x1_x3"] = num_df["x1"] * num_df["x3"]
num_df["x2_x3"] = num_df["x2"] * num_df["x3"]

train_n, test_n = train_test_split(num_df, test_size=0.2, random_state=RANDOM_STATE)
poly = PolynomialFeatures(degree=2, include_bias=False)
train_poly = poly.fit_transform(train_n[["x1", "x2", "x3"]])
test_poly = poly.transform(test_n[["x1", "x2", "x3"]])

num_df["x1_bin"] = pd.qcut(num_df["x1"], q=4, labels=False)

# 2.3
agg = tx.groupby("customer_id").agg(
    total_spent=("amount", "sum"),
    avg_check=("amount", "mean"),
    n_transactions=("amount", "count"),
    n_unique_categories=("category", "nunique"),
    max_amount=("amount", "max"),
).reset_index()
df_enriched = df.merge(agg, on="customer_id", how="left")
'''


---
# Раздел 3. Регрессия


## Задача 3.1 — Линейная регрессия с регуляризацией

**Требуется:**
1. Разбить данные на train/test.
2. Обучить `LinearRegression`, `Ridge`, `Lasso`.
3. Посчитать MAE, RMSE, R² на test для каждой модели.
4. Сравнить коэффициенты Lasso — какие признаки "обнулились"?


In [ ]:
X_reg, y_reg = make_regression(n_samples=500, n_features=10, n_informative=5,
                                noise=15, random_state=RANDOM_STATE)
X_reg = pd.DataFrame(X_reg, columns=[f"f{i}" for i in range(10)])
y_reg = pd.Series(y_reg, name="target")
X_reg.head()


In [ ]:
# TODO 3.1: train/test split, обучите LinearRegression/Ridge/Lasso,
#           посчитайте MAE/RMSE/R^2, сравните коэффициенты Lasso


## Задача 3.2 — Подбор гиперпараметров и нелинейная модель

Обучите `RandomForestRegressor` с подбором `n_estimators` и `max_depth` через `GridSearchCV`
(5-fold CV, метрика — neg RMSE). Сравните с результатом линейной регрессии из 3.1.
Выведите важность признаков (`feature_importances_`).


In [ ]:
# TODO 3.2: GridSearchCV для RandomForestRegressor + сравнение с 3.1 + feature_importances_


In [ ]:
# ===================== РЕШЕНИЕ раздела 3 (скрыто) =====================
'''
X_train, X_test, y_train, y_test = train_test_split(X_reg, y_reg, test_size=0.2, random_state=RANDOM_STATE)

models = {"linreg": LinearRegression(), "ridge": Ridge(alpha=1.0), "lasso": Lasso(alpha=0.5)}
for name, m in models.items():
    m.fit(X_train, y_train)
    pred = m.predict(X_test)
    mae = mean_absolute_error(y_test, pred)
    rmse = mean_squared_error(y_test, pred, squared=False)
    r2 = r2_score(y_test, pred)
    print(name, "MAE:", round(mae,2), "RMSE:", round(rmse,2), "R2:", round(r2,3))

print("Lasso coefs:", dict(zip(X_reg.columns, models["lasso"].coef_.round(2))))

# 3.2
param_grid = {"n_estimators": [100, 200], "max_depth": [3, 5, None]}
gs = GridSearchCV(RandomForestRegressor(random_state=RANDOM_STATE), param_grid,
                   cv=5, scoring="neg_root_mean_squared_error")
gs.fit(X_train, y_train)
print(gs.best_params_)
best_rf = gs.best_estimator_
print("RF RMSE:", mean_squared_error(y_test, best_rf.predict(X_test), squared=False))
print(dict(zip(X_reg.columns, best_rf.feature_importances_.round(3))))
'''


---
# Раздел 4. Бинарная классификация


## Задача 4.1 — Логистическая регрессия и метрики

Датасет несбалансирован (10% положительного класса — например, "клиент ушёл").

**Требуется:**
1. Обучить `LogisticRegression` (с `class_weight='balanced'`).
2. Посчитать accuracy, precision, recall, f1, ROC-AUC.
3. Построить confusion matrix и объяснить, почему accuracy — плохая метрика здесь.
4. Подобрать оптимальный порог классификации (threshold) вместо дефолтных 0.5, максимизируя F1.


In [ ]:
X_clf, y_clf = make_classification(n_samples=1000, n_features=15, n_informative=6,
                                    weights=[0.9, 0.1], random_state=RANDOM_STATE)
X_clf = pd.DataFrame(X_clf, columns=[f"f{i}" for i in range(15)])
y_clf = pd.Series(y_clf, name="churn")
print(y_clf.value_counts(normalize=True))


In [ ]:
# TODO 4.1.1: train/test split (stratify=y!), обучите LogisticRegression(class_weight='balanced')


# TODO 4.1.2: accuracy, precision, recall, f1, roc_auc на test


# TODO 4.1.3: confusion_matrix, classification_report


# TODO 4.1.4: переберите threshold от 0.1 до 0.9 (predict_proba), найдите threshold с максимальным F1


## Задача 4.2 — Сравнение моделей через ROC-кривую

Обучите `LogisticRegression`, `RandomForestClassifier`, `GradientBoostingClassifier`.
Постройте ROC-кривые всех трёх на одном графике, сравните AUC.


In [ ]:
# TODO 4.2: обучите 3 модели, постройте ROC-кривые (fpr, tpr = roc_curve(...)) на одном plt.figure


In [ ]:
# ===================== РЕШЕНИЕ раздела 4 (скрыто) =====================
'''
from sklearn.metrics import roc_curve

X_train, X_test, y_train, y_test = train_test_split(X_clf, y_clf, test_size=0.2,
                                                      stratify=y_clf, random_state=RANDOM_STATE)
logreg = LogisticRegression(class_weight="balanced", max_iter=1000)
logreg.fit(X_train, y_train)
proba = logreg.predict_proba(X_test)[:, 1]
pred = logreg.predict(X_test)

print("accuracy", accuracy_score(y_test, pred))
print("precision", precision_score(y_test, pred))
print("recall", recall_score(y_test, pred))
print("f1", f1_score(y_test, pred))
print("roc_auc", roc_auc_score(y_test, proba))
print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred))

best_f1, best_t = 0, 0.5
for t in np.arange(0.1, 0.91, 0.01):
    p = (proba >= t).astype(int)
    f1 = f1_score(y_test, p)
    if f1 > best_f1:
        best_f1, best_t = f1, t
print("best threshold", best_t, "f1", best_f1)

# 4.2
models = {
    "logreg": LogisticRegression(class_weight="balanced", max_iter=1000),
    "rf": RandomForestClassifier(random_state=RANDOM_STATE),
    "gb": GradientBoostingClassifier(random_state=RANDOM_STATE),
}
plt.figure()
for name, m in models.items():
    m.fit(X_train, y_train)
    p = m.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, p)
    plt.plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_test, p):.3f})")
plt.plot([0,1],[0,1],"--", color="gray")
plt.legend(); plt.xlabel("FPR"); plt.ylabel("TPR"); plt.show()
'''


---
# Раздел 5. Мультиклассовая классификация


## Задача 5.1 — Мультиклассовая классификация и confusion matrix

Датасет с 4 классами.

**Требуется:**
1. Обучить `LogisticRegression(multi_class='multinomial')` и `RandomForestClassifier`.
2. Посчитать accuracy, macro-F1, weighted-F1.
3. Построить confusion matrix (4x4), определить, какие классы модель путает чаще всего.


In [ ]:
X_mc, y_mc = make_classification(n_samples=1200, n_features=12, n_informative=8,
                                  n_classes=4, n_clusters_per_class=1, random_state=RANDOM_STATE)
X_mc = pd.DataFrame(X_mc, columns=[f"f{i}" for i in range(12)])
y_mc = pd.Series(y_mc, name="label")
y_mc.value_counts()


In [ ]:
# TODO 5.1.1: train/test split (stratify), обучите LogisticRegression и RandomForestClassifier


# TODO 5.1.2: accuracy, f1_score(average='macro'), f1_score(average='weighted') для обеих моделей


# TODO 5.1.3: confusion_matrix 4x4, найдите пару классов с наибольшей путаницей


## Задача 5.2 — One-vs-Rest ROC-AUC для мультикласса

Посчитайте ROC-AUC для мультиклассовой задачи через `roc_auc_score(..., multi_class='ovr')`,
используя `predict_proba`. Сравните `ovr` и `ovo` стратегии.


In [ ]:
# TODO 5.2: roc_auc_score с multi_class='ovr' и 'ovo', сравните значения


In [ ]:
# ===================== РЕШЕНИЕ раздела 5 (скрыто) =====================
'''
X_train, X_test, y_train, y_test = train_test_split(X_mc, y_mc, test_size=0.2,
                                                      stratify=y_mc, random_state=RANDOM_STATE)
logreg = LogisticRegression(multi_class="multinomial", max_iter=1000)
rf = RandomForestClassifier(random_state=RANDOM_STATE)

for name, m in [("logreg", logreg), ("rf", rf)]:
    m.fit(X_train, y_train)
    pred = m.predict(X_test)
    print(name, "acc", accuracy_score(y_test, pred),
          "macro-f1", f1_score(y_test, pred, average="macro"),
          "weighted-f1", f1_score(y_test, pred, average="weighted"))
    print(confusion_matrix(y_test, pred))

proba = rf.predict_proba(X_test)
print("ovr", roc_auc_score(y_test, proba, multi_class="ovr"))
print("ovo", roc_auc_score(y_test, proba, multi_class="ovo"))
'''


---
# Раздел 6. Кластеризация


## Задача 6.1 — KMeans и выбор числа кластеров

**Требуется:**
1. Стандартизировать признаки.
2. Методом "локтя" (elbow method, inertia) и silhouette score подобрать оптимальное k (от 2 до 8).
3. Обучить KMeans с лучшим k, визуализировать кластеры (если признаков > 2 — через первые 2 главные компоненты PCA или просто 2 первых признака).


In [ ]:
X_cl, y_true = make_blobs(n_samples=600, centers=5, n_features=6, cluster_std=1.8,
                           random_state=RANDOM_STATE)
X_cl = pd.DataFrame(X_cl, columns=[f"f{i}" for i in range(6)])
X_cl.head()


In [ ]:
# TODO 6.1.1: StandardScaler


# TODO 6.1.2: для k в range(2, 9) посчитайте inertia_ и silhouette_score, постройте графики


# TODO 6.1.3: обучите KMeans с лучшим k, визуализируйте (PCA до 2 компонент или 2 признака)


## Задача 6.2 — Сравнение алгоритмов кластеризации

Сравните `KMeans`, `AgglomerativeClustering`, `DBSCAN` на одних и тех же данных
по silhouette score и davies_bouldin_score. Для DBSCAN подберите `eps` (например, через
k-distance график или перебором нескольких значений).


In [ ]:
# TODO 6.2: сравните KMeans / AgglomerativeClustering / DBSCAN по silhouette и davies_bouldin


In [ ]:
# ===================== РЕШЕНИЕ раздела 6 (скрыто) =====================
'''
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cl)

inertias, sils = [], []
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sils.append(silhouette_score(X_scaled, labels))

fig, ax = plt.subplots(1, 2, figsize=(10,4))
ax[0].plot(range(2,9), inertias, marker="o"); ax[0].set_title("Elbow (inertia)")
ax[1].plot(range(2,9), sils, marker="o"); ax[1].set_title("Silhouette score")
plt.show()

best_k = list(range(2,9))[int(np.argmax(sils))]
km = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=10)
labels = km.fit_predict(X_scaled)

from sklearn.decomposition import PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
plt.scatter(X_pca[:,0], X_pca[:,1], c=labels, cmap="tab10")
plt.title(f"KMeans k={best_k}"); plt.show()

# 6.2
agg = AgglomerativeClustering(n_clusters=best_k).fit_predict(X_scaled)
db = DBSCAN(eps=1.2, min_samples=5).fit_predict(X_scaled)

for name, lab in [("kmeans", labels), ("agglo", agg), ("dbscan", db)]:
    mask = lab != -1  # для DBSCAN исключаем шум при оценке
    if len(set(lab[mask])) > 1:
        print(name, "silhouette", silhouette_score(X_scaled[mask], lab[mask]),
              "davies_bouldin", davies_bouldin_score(X_scaled[mask], lab[mask]))
'''


---
## Что дальше

- Попробуйте пройти все задачи на время (таймер 15 мин/задача) — это ближе к формату реального live-coding интервью.
- После решения — сверьтесь с ячейками "РЕШЕНИЕ", но помните, что вариантов правильного решения обычно несколько.
- Если нужно — попросите отдельный ноутбук с задачами на конкретную тему (например, только про работу с пайплайнами `Pipeline`+`ColumnTransformer`, или про временные ряды).
